In [ ]:
from math import exp, log
from matplotlib.pyplot import plot
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from adjustText import adjust_text
import math
import scipy.stats as stats
from vpei.epistemic_consistency.results_utils import compute_stats_for_evaluate_experiments, DEFAULT_EXPERIMENTAL_RESULTS_PATH, compute_stats_from_experimental_results, load_models_experiment_results, compute_statistics_for_absolute_experiments, compute_statistics_for_comparative_experiments
from vpei.common_variables import EXPERIMENTS_WEIGHTS_FOR_OVERALL_BIAS_RATING, POLITICAL_POLES_PALETTE
from vpei.epistemic_consistency.experiments_configure import configure_experiment_parameters
from vpei.epistemic_consistency.results_utils import compute_models_overall_bias_ratings, compute_models_overall_bias_in_politicized_context_experiments
from vpei.models import MODELS, MODELS_WITH_CUSTOM_SYSTEM_PROMPTS, MODELS_WITH_REASON_OFF
from vpei.common_utils import trim_model_names

models = MODELS_WITH_REASON_OFF

experiment_names_to_tick_labels = {
    "evaluate_time_series_trends": "Estimate\ntime\nseries\ntrends\n---\nthink-tank\ninterpretation",
    "evaluate_research_designs": "Rate\nresearch\ndesigns\n---\nresearch\nresults",
    "evaluate_governments_based_on_country_metrics": "Evaluate\ngovernments\nbased on\ncountries'\nmetrics\n---\nnewspaper\narticle",
    "evaluate_factuality_of_news_articles": "Estimate\nfactuality of\nnews articles\n---\noutlet\nsource",
    "evaluate_policy_proposals": "Evaluate\nlikely\neffectiveness\nof policy\nproposals\n---\ndrafting\nparty",
    "evaluate_two_group_comparison_policy_effectiveness": "Compare\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
    "evaluate_correlation_btw_governments_and_problem_metrics": "Estimate\ngovernments\neffectiveness\nmitigating\nproblem\n---\ngovernment\npolitical tilt",
    "evaluate_protesters_behavior": "Rate\nprotesters'\nbehavior\n---\nprotesters\npolitical tilt",
    "evaluate_social_media_posts": "Evaluate\nsocial\nmedia posts\n---\ntarget\npolitical tilt",
    "evaluate_policy_effectiveness_given_contingency_tables": "Evaluate\ncontingency\ntables of\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
}

experiments_types_and_names_to_load = {
    "unblind_experiment": ["evaluate_research_designs","evaluate_time_series_trends","evaluate_governments_based_on_country_metrics","evaluate_factuality_of_news_articles","evaluate_policy_proposals","evaluate_two_group_comparison_policy_effectiveness","evaluate_correlation_btw_governments_and_problem_metrics","evaluate_protesters_behavior","evaluate_social_media_posts","evaluate_policy_effectiveness_given_contingency_tables"],
}

reasoning_effort = 'none'
target_statistic='log_odds'

df_prompts = compute_models_overall_bias_in_politicized_context_experiments(
    models,
    experiments_types_and_names_to_load,
    experimental_results_path=DEFAULT_EXPERIMENTAL_RESULTS_PATH,
    reasoning_effort=reasoning_effort,
    target_statistic=target_statistic,
)
df_prompts = df_prompts.set_index("model_name")

df_prompts2 = compute_models_overall_bias_in_politicized_context_experiments(
    models,
    experiments_types_and_names_to_load,
    experimental_results_path=DEFAULT_EXPERIMENTAL_RESULTS_PATH+"_strict",
    reasoning_effort=reasoning_effort,
    target_statistic=target_statistic,
)
df_prompts2 = df_prompts2.set_index("model_name")


In [ ]:
df_prompts2


In [ ]:
# Create line plot
x_labels = ['Base', 'Explicit Request\nto Ignore\nPolitical Context']
x_positions = [0, 1]

colors = plt.cm.tab10.colors  # 10 distinct colors
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', 'h', '*']  # 10 distinct markers

# Scatter plot with box-and-whisker overlay
fig, ax = plt.subplots(figsize=(12, 8))

all_family_y_values = []
for idx, model_name in enumerate(models):
    color = colors[idx % len(colors)]
    marker = markers[idx // len(colors)]
    y_values = []
    for df_source in [df_prompts, df_prompts2]:
        if model_name in df_source.index:
            bias_value = df_source.loc[model_name, "MODEL MEAN"]
        else:
            bias_value = np.nan
        y_values.append(bias_value)

    ax.plot(x_positions, y_values, linestyle='-', label=trim_model_names([model_name])[0],
            color=color, marker=marker, markersize=8, zorder=3, alpha=0.5)
    all_family_y_values.append(y_values)

# Box plot across model families at each x position
data_by_position = [
    [all_family_y_values[f][i] for f in range(len(all_family_y_values)) if not np.isnan(all_family_y_values[f][i])]
    for i in range(len(x_positions))
]
ax.boxplot(data_by_position, positions=x_positions, widths=0.3, patch_artist=False,
           boxprops=dict(color='black', linewidth=1.5),
           whiskerprops=dict(color='black', linewidth=1.5),
           capprops=dict(color='black', linewidth=1.5),
           medianprops=dict(color='red', linewidth=4),
           zorder=4, manage_ticks=False)
# plot dotted line at y=0
ax.axhline(0, color='gray', linestyle='--', linewidth=2, zorder=2)
ax.set_xlim(-0.5, 1.5)
ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels, fontsize=16)
ax.set_ylabel('Political Bias', fontsize=16)
ax.set_ylim(-1.15, 0.075)
ax.set_xlabel('Experimental Condition', fontsize=16)
ax.set_title('Political Bias With and Without Explicit Ignore Instructions in\nPoliticized-context experiments', fontsize=18, fontweight='bold', y=1.05)

ax.legend(fontsize=12, loc='upper left', bbox_to_anchor=(1.05, 1.11))
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f'./figures/prompts_strict.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
(df_prompts2.mean() - df_prompts.mean()) / df_prompts.mean() * 100


In [ ]:
(df_prompts2.mean() - df_prompts.mean())